# 40 - Reto 1: Chatbot Design (Capa de Orquestacion Conversacional)

## Proposito
Definir la capa conversacional/orquestadora del sistema analitico de Reto 1. Este notebook no rehace calculos de NB20 o NB30: diseña como el chatbot interpreta preguntas, decide herramientas, aplica reglas y responde con evidencia y limites.

## Relacion con NB20 y NB30
- NB20 aporta contrato semantico, intents, comparabilidad y reglas de lenguaje.
- NB30 aporta candidatos de insight, scoring, caveats y narrativa templada.
- NB40 define como el chatbot consume esas capas sin inventar calculos.

## Outputs esperados
- docs/architecture/reto1_chatbot_contract.md
- reports/reto1/chatbot_design_summary.md
- reports/reto1/golden_conversation_flows.md
- reports/reto1/chatbot_planner_schema.json (opcional util para implementacion)

## 0) Setup y contexto

### Que se hace
Se cargan contratos semanticos, reglas de negocio, intents y outputs del insight engine para diseñar la capa conversacional gobernada.

### Por que se hace
El diseño del chatbot debe operar sobre artefactos auditables del sistema analitico, no sobre inferencias libres del LLM.

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json
import textwrap

import pandas as pd
import yaml

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / 'config').exists() and (p / 'reports' / 'reto1').exists():
            return p
    raise FileNotFoundError('No se encontro root del proyecto.')

ROOT = find_project_root()
CONFIG_DIR = ROOT / 'config'
REPORTS_DIR = ROOT / 'reports' / 'reto1'
ARCH_DIR = ROOT / 'docs' / 'architecture'
ARCH_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_DIR / 'metrics.yaml', 'r', encoding='utf-8') as f:
    metrics_cfg = yaml.safe_load(f)
with open(CONFIG_DIR / 'business_rules.yaml', 'r', encoding='utf-8') as f:
    business_rules = yaml.safe_load(f)
with open(CONFIG_DIR / 'question_types.yaml', 'r', encoding='utf-8') as f:
    question_types_cfg = yaml.safe_load(f)

semantic_report = json.loads((REPORTS_DIR / 'semantic_layer_report.json').read_text(encoding='utf-8')) if (REPORTS_DIR / 'semantic_layer_report.json').exists() else {}
insight_report = json.loads((REPORTS_DIR / 'insight_engine_report.json').read_text(encoding='utf-8')) if (REPORTS_DIR / 'insight_engine_report.json').exists() else {}
insight_candidates = pd.read_parquet(REPORTS_DIR / 'insight_candidates.parquet') if (REPORTS_DIR / 'insight_candidates.parquet').exists() else pd.DataFrame()

metrics_catalog = pd.DataFrame(metrics_cfg.get('metrics', []))
intents_catalog = pd.DataFrame(question_types_cfg.get('question_types', []))

print('ROOT:', ROOT)
print('intents:', intents_catalog.shape[0])
print('metrics:', metrics_catalog.shape[0])
print('insight_candidates:', insight_candidates.shape)

## 1) Arquitectura conversacional propuesta

### Que hace el chatbot
- Interpreta intencion del usuario y extrae parametros.
- Orquesta llamadas a funciones deterministicas.
- Ensambla respuesta en formato estable con evidencia y caveats.
- Gestiona contexto conversacional minimo para follow-ups.

### Que NO hace el chatbot
- No calcula metricas sobre datos crudos.
- No reemplaza semantic layer ni insight engine.
- No afirma causalidad con evidencia asociativa.
- No ignora reglas de comparabilidad o metricas suspendidas.

### Papel del LLM
El LLM se usa como planner/orquestador y sintetizador controlado. El calculo permanece fuera del LLM en herramientas deterministicas.

## 2) Catalogo de intents operativos

Mapeo de intents semanticos (NB20) a comportamiento conversacional y de orquestacion.

In [ ]:
def list_to_text(x):
    if isinstance(x, list):
        return ', '.join(str(v) for v in x)
    return str(x) if x is not None else ''

intents_ops = intents_catalog.copy()
intents_ops['required_params_text'] = intents_ops.get('required_params', pd.Series([[]] * len(intents_ops))).apply(list_to_text)
intents_ops['optional_params_text'] = intents_ops.get('optional_params', pd.Series([[]] * len(intents_ops))).apply(list_to_text)
intents_ops['future_function_name'] = intents_ops.get('future_function', '')
intents_ops['requires_visualization'] = intents_ops.get('default_visualization', '').astype(str).ne('')
intents_ops['direct_answer_possible'] = intents_ops['id'].isin(['query'])
intents_ops['requires_clarification_by_default'] = intents_ops['id'].isin(['multivariable_filter', 'hypothesis_request'])
intents_ops['requires_insight_engine'] = intents_ops['id'].isin(['insight_request'])
intents_ops['hypothesis_mode'] = intents_ops['id'].isin(['hypothesis_request'])

intent_view = intents_ops[[
    'id', 'display_name', 'support_level',
    'required_params_text', 'optional_params_text',
    'future_function_name', 'requires_visualization',
    'direct_answer_possible', 'requires_clarification_by_default',
    'requires_insight_engine', 'hypothesis_mode',
]].rename(columns={'id': 'intent'})

display(intent_view)

## 3) Diseño del planner estructurado

El planner transforma texto libre a un plan ejecutable y validable antes de llamar herramientas.

In [ ]:
planner_schema = {
    'title': 'ChatbotExecutionPlan',
    'type': 'object',
    'required': ['intent', 'requested_output', 'requires_clarification', 'confidence_mode'],
    'properties': {
        'intent': {'type': 'string', 'description': 'Intent canonico de question_types'},
        'metric': {'type': ['string', 'null']},
        'entity_scope': {
            'type': 'object',
            'properties': {
                'country': {'type': ['string', 'null']},
                'city': {'type': ['string', 'null']},
                'zone': {'type': ['string', 'null']},
                'segment': {'type': ['string', 'null']},
            },
            'additionalProperties': False,
        },
        'filters': {'type': 'array', 'items': {'type': 'object'}},
        'comparison': {'type': ['object', 'null']},
        'time_window': {'type': ['string', 'null'], 'description': 'ej: L0W, L8W-L0W'},
        'requested_output': {'type': 'string', 'enum': ['kpi', 'table', 'chart', 'narrative', 'insights']},
        'chart_preference': {'type': ['string', 'null']},
        'confidence_mode': {'type': 'string', 'enum': ['strict', 'balanced', 'exploratory']},
        'requires_clarification': {'type': 'boolean'},
        'clarification_question': {'type': ['string', 'null']},
        'tool_calls': {'type': 'array', 'items': {'type': 'object'}},
    },
    'additionalProperties': False,
}

planner_examples = pd.DataFrame([
    {
        'user_question': 'Compara Perfect Orders entre Wealthy y Non Wealthy en Mexico',
        'intent': 'compare',
        'metric': 'perfect_orders',
        'entity_scope': {'country': 'MX', 'segment': 'ZONE_TYPE'},
        'time_window': 'L0W',
        'requested_output': 'table',
        'requires_clarification': False,
    },
    {
        'user_question': 'Que explica el crecimiento de orders en Chapinero?',
        'intent': 'hypothesis_request',
        'metric': 'orders_total',
        'entity_scope': {'country': 'CO', 'city': 'Bogota', 'zone': 'Chapinero'},
        'time_window': 'L8W-L0W',
        'requested_output': 'narrative',
        'requires_clarification': False,
    },
    {
        'user_question': 'Top 5 zonas con mayor lead penetration',
        'intent': 'rank',
        'metric': 'lead_penetration',
        'entity_scope': {},
        'time_window': 'L0W',
        'requested_output': 'table',
        'requires_clarification': True,
    },
])

display(pd.DataFrame([{'schema': json.dumps(planner_schema, ensure_ascii=False)[:500] + '...'}]))
display(planner_examples)

## 4) Reglas de clarificacion

El chatbot no debe asumir parametros ambiguos ni ejecutar comparaciones invalidas segun NB20.

In [ ]:
clarification_rules = pd.DataFrame([
    {'trigger': 'metrica_ambigua', 'why': 'No se puede ejecutar funcion sin metrica canonica', 'good_clarification': 'Que metrica quieres analizar? Ej: Perfect Orders o Gross Profit UE.'},
    {'trigger': 'entidad_ambigua', 'why': 'ZONE no es unica sin COUNTRY y CITY', 'good_clarification': 'Te refieres a que pais y ciudad para esa zona?'},
    {'trigger': 'comparacion_invalida', 'why': 'NB20 define pares no comparables', 'good_clarification': 'Esa comparacion no es valida. Quieres comparar dentro del mismo pais y tipo de zona?'},
    {'trigger': 'peer_group_debil', 'why': 'Benchmark con n pequeno es fragil', 'good_clarification': 'El peer group es pequeno. Continuo con caveat o prefieres ampliar el alcance?'},
    {'trigger': 'causalidad_fuerte', 'why': 'El sistema solo soporta asociaciones no causales', 'good_clarification': 'Puedo darte posibles drivers asociados, no causalidad confirmada. Continuo?'},
    {'trigger': 'fuera_de_alcance', 'why': 'Intent no cubierto por funciones deterministicas', 'good_clarification': 'No puedo responder eso con el contrato actual. Te propongo reformular en formato ranking, tendencia, comparacion o insight.'},
    {'trigger': 'metrica_suspendida', 'why': 'validation_status suspendida impide uso analitico confiable', 'good_clarification': 'Esa metrica esta suspendida temporalmente. Quieres usar una metrica alternativa?'},
])

display(clarification_rules)

## 5) Gestion de contexto conversacional

Se define estado minimo, seguro y util para follow-ups sin memoria libre peligrosa.

In [ ]:
chat_state_schema = {
    'title': 'ChatSessionState',
    'type': 'object',
    'properties': {
        'last_metric_id': {'type': ['string', 'null']},
        'last_entity': {'type': ['object', 'null']},
        'last_filters': {'type': 'array', 'items': {'type': 'object'}},
        'last_comparison': {'type': ['object', 'null']},
        'last_time_window': {'type': ['string', 'null']},
        'mode': {'type': 'string', 'enum': ['direct_answer', 'hypothesis_mode']},
        'last_visualization': {'type': ['string', 'null']},
    },
    'required': ['mode', 'last_filters'],
    'additionalProperties': False,
}

context_policy = pd.DataFrame([
    {'keep': 'last_metric_id', 'reason': 'permite resolver follow-up tipo "y en Colombia?"'},
    {'keep': 'last_entity', 'reason': 'evita pedir entidad completa en cada turno'},
    {'keep': 'last_filters', 'reason': 'mantiene alcance analitico trazable'},
    {'keep': 'last_comparison', 'reason': 'preserva baseline de comparacion'},
    {'keep': 'last_time_window', 'reason': 'evita cambios silenciosos de periodo'},
    {'keep': 'mode', 'reason': 'separa respuesta directa de hypothesis mode'},
    {'do_not_keep': 'supuestos_no_confirmados', 'reason': 'evita memoria inventada'},
    {'do_not_keep': 'narrativas_largas', 'reason': 'reduce arrastre de contexto irrelevante'},
])

display(context_policy.fillna(''))

## 6) Contrato de herramientas / funciones

Las funciones son deterministicas y ejecutan el calculo. El chatbot solo planifica, valida y presenta resultados.

In [ ]:
tool_contract = pd.DataFrame([
    {'tool': 'get_metric_value', 'input_schema': 'metric_id, entity_scope, week_offset', 'output_schema': 'value, metadata, caveats', 'errors': 'metric_not_found|entity_not_found|insufficient_data', 'chatbot_caveat': 'direction/validation caveats'},
    {'tool': 'aggregate_metric', 'input_schema': 'metric_id, group_by, filters, week_offset, agg_func', 'output_schema': 'rows[group, value, n, coverage]', 'errors': 'invalid_group_by|unsupported_metric', 'chatbot_caveat': 'coverage and excluded zones'},
    {'tool': 'rank_by_metric', 'input_schema': 'metric_id, entity_level, n, direction, filters', 'output_schema': 'ranked_rows', 'errors': 'metric_not_rankable|direction_mismatch', 'chatbot_caveat': 'invert when lower_is_better'},
    {'tool': 'get_trend', 'input_schema': 'metric_id, entity_scope, window', 'output_schema': 'series[value,wow]', 'errors': 'insufficient_history', 'chatbot_caveat': 'no complex forecasting with 9 weeks'},
    {'tool': 'compare_segments', 'input_schema': 'metric_id, segment_a, segment_b, week_offset', 'output_schema': 'comparison_table + delta', 'errors': 'invalid_comparison', 'chatbot_caveat': 'respect not_comparable rules'},
    {'tool': 'screen_by_conditions', 'input_schema': 'conditions[], entity_level, week_offset', 'output_schema': 'matching_entities', 'errors': 'missing_threshold|unsupported_condition', 'chatbot_caveat': 'thresholds must be explicit'},
    {'tool': 'run_insight_detectors', 'input_schema': 'entity_scope, metrics, detectors, week_offset', 'output_schema': 'insight_candidates[]', 'errors': 'detector_not_available', 'chatbot_caveat': 'provisional thresholds caveat'},
    {'tool': 'get_insight_candidates', 'input_schema': 'filters, sort, limit', 'output_schema': 'insight_table', 'errors': 'empty_result', 'chatbot_caveat': 'report confidence and caveats'},
    {'tool': 'generate_hypothesis_candidates', 'input_schema': 'target_metric, entity_scope, candidate_metrics', 'output_schema': 'association_table', 'errors': 'insufficient_points|target_not_supported', 'chatbot_caveat': 'association not causation'},
    {'tool': 'render_chart_data', 'input_schema': 'intent, data_payload, chart_pref', 'output_schema': 'chart_spec', 'errors': 'chart_not_supported', 'chatbot_caveat': 'chart is view, not evidence source'},
])

display(tool_contract)

## 7) Contrato de respuesta del chatbot

Formato fijo por respuesta para evitar texto libre desordenado:
1. respuesta corta
2. evidencia principal
3. filtros/alcance usados
4. visualizacion sugerida o incluida
5. caveat/limite
6. siguientes preguntas sugeridas

In [ ]:
response_contract = pd.DataFrame([
    {'intent': 'query', 'short_answer': 'valor puntual', 'evidence': 'valor + semana + fuente', 'scope': 'entity + metric + offset', 'viz': 'kpi_card', 'caveat': 'validation/direction', 'next_questions': 'trend|compare'},
    {'intent': 'aggregate', 'short_answer': 'estadistico principal', 'evidence': 'tabla agregada + n', 'scope': 'group_by + filtros', 'viz': 'bar_chart', 'caveat': 'coverage/missing', 'next_questions': 'rank|segment drill-down'},
    {'intent': 'rank', 'short_answer': 'top/bottom n', 'evidence': 'tabla rankeada', 'scope': 'entity_level + n + filtros', 'viz': 'ranked_table', 'caveat': 'lower_is_better + elegibilidad', 'next_questions': 'why this zone?|trend'},
    {'intent': 'trend', 'short_answer': 'direccion reciente', 'evidence': 'serie + wow', 'scope': 'entity + ventana', 'viz': 'line_chart', 'caveat': '9 weeks no forecasting', 'next_questions': 'insight_request'},
    {'intent': 'compare', 'short_answer': 'delta entre segmentos', 'evidence': 'tabla comparativa + n', 'scope': 'segment_a vs segment_b', 'viz': 'side_by_side_bar', 'caveat': 'comparabilidad', 'next_questions': 'rank within segment'},
    {'intent': 'insight_request', 'short_answer': 'top hallazgos', 'evidence': 'insight_candidates ordenados', 'scope': 'entity/filtro + detectores', 'viz': 'alert_cards', 'caveat': 'thresholds provisionales', 'next_questions': 'show evidence|actions'},
    {'intent': 'hypothesis_request', 'short_answer': 'posibles drivers', 'evidence': 'asociaciones y caveats', 'scope': 'target_metric + entidad', 'viz': 'correlation_table', 'caveat': 'no causalidad', 'next_questions': 'validate with ops context'},
])

display(response_contract)

## 8) Reglas de lenguaje y seguridad analitica

Se operacionalizan reglas de NB20 para respuestas conversacionales consistentes y seguras.

In [ ]:
language_ops = pd.DataFrame([
    {'rule': 'mejoro/empeoro', 'when_allowed': 'solo con desired_direction definida', 'required_caveat': 'direction provisional si aplica'},
    {'rule': 'possible_driver', 'when_allowed': 'solo en intent hypothesis_request', 'required_caveat': 'association_not_causation'},
    {'rule': 'causality_guard', 'when_allowed': 'si usuario pide causalidad', 'required_caveat': 'rechazar causalidad fuerte y ofrecer hipotesis'},
    {'rule': 'peer_weak', 'when_allowed': 'peer_confidence low_confidence', 'required_caveat': 'benchmark fragil por n pequeno'},
    {'rule': 'metric_pending_validation', 'when_allowed': 'validation_status pending', 'required_caveat': 'resultado sujeto a validacion negocio'},
    {'rule': 'metric_suspended', 'when_allowed': 'validation_status suspended', 'required_caveat': 'no usar; proponer alternativa'},
])

display(language_ops)

## 9) UX conversacional propuesta

Wireframe logico de respuesta por tipo de pregunta.

- Header: pregunta normalizada + alcance
- Body: respuesta corta + evidencia + tabla/grafico
- Footer: caveats + prompts sugeridos

In [ ]:
ux_components = pd.DataFrame([
    {'intent': 'query', 'primary_component': 'kpi_card', 'secondary_component': 'peer_context_chip', 'suggested_prompts': 'Ver tendencia | Comparar vs peers'},
    {'intent': 'compare', 'primary_component': 'comparison_table', 'secondary_component': 'delta_bar', 'suggested_prompts': 'Solo en Colombia | Mostrar top gaps'},
    {'intent': 'trend', 'primary_component': 'line_chart', 'secondary_component': 'wow_table', 'suggested_prompts': 'Mostrar solo ultimas 4 semanas | Activar alertas'},
    {'intent': 'insight_request', 'primary_component': 'alert_cards', 'secondary_component': 'evidence_panel', 'suggested_prompts': 'Ver evidencia | Priorizar por severidad'},
    {'intent': 'hypothesis_request', 'primary_component': 'association_table', 'secondary_component': 'caveat_panel', 'suggested_prompts': 'Ver solo asociaciones fuertes | Mostrar no causalidad'},
])

wireframe_text = textwrap.dedent('''
[Conversation Header]
  - User query normalized
  - Active scope chips (metric, entity, period)

[Primary Insight Block]
  - Short answer
  - Main evidence table/chart

[Governance Block]
  - Caveats (validation, peer confidence, causality guard)
  - Tool trace (optional for audit mode)

[Follow-up Block]
  - Suggested prompts
''')

display(ux_components)
print(wireframe_text)

## 10) Golden conversation flows

Se definen flujos esperados (intent -> herramientas -> respuesta -> caveat) para pruebas end-to-end del orquestador.

In [ ]:
golden_flows = pd.DataFrame([
    {'flow_id': 'GF01', 'user_query': 'Top 5 zonas con mayor Lead Penetration', 'intent': 'rank', 'tools': 'rank_by_metric', 'requires_clarification': True, 'ideal_response': 'rechazar metrica suspendida y proponer alternativa', 'expected_caveat': 'metric_suspended'},
    {'flow_id': 'GF02', 'user_query': 'Compara Perfect Orders entre Wealthy y Non Wealthy en Mexico', 'intent': 'compare', 'tools': 'compare_segments', 'requires_clarification': False, 'ideal_response': 'tabla comparativa con delta', 'expected_caveat': 'direction provisional'},
    {'flow_id': 'GF03', 'user_query': 'Evolucion de Gross Profit UE en Chapinero', 'intent': 'trend', 'tools': 'get_trend, render_chart_data', 'requires_clarification': True, 'ideal_response': 'pedir ciudad/pais si falta', 'expected_caveat': 'no forecasting con 9 semanas'},
    {'flow_id': 'GF04', 'user_query': 'Promedio de Lead Penetration por pais', 'intent': 'aggregate', 'tools': 'aggregate_metric', 'requires_clarification': True, 'ideal_response': 'rechazar uso por suspension', 'expected_caveat': 'metric_suspended'},
    {'flow_id': 'GF05', 'user_query': 'Zonas con alto Lead Penetration y bajo Perfect Orders', 'intent': 'multivariable_filter', 'tools': 'screen_by_conditions', 'requires_clarification': True, 'ideal_response': 'pedir reemplazo de metrica suspendida y thresholds', 'expected_caveat': 'unsupported_condition'},
    {'flow_id': 'GF06', 'user_query': 'Zonas que mas crecen en orders y que podria explicarlo', 'intent': 'hypothesis_request', 'tools': 'get_insight_candidates, generate_hypothesis_candidates', 'requires_clarification': False, 'ideal_response': 'drivers asociados no causales', 'expected_caveat': 'association_not_causation'},
    {'flow_id': 'GF07', 'user_query': 'Y solo en Colombia?', 'intent': 'follow_up_scope_refine', 'tools': 'reuse_last_tool_with_filter', 'requires_clarification': False, 'ideal_response': 'mantener metrica e intent previos filtrando country=CO', 'expected_caveat': 'preserve prior context'},
    {'flow_id': 'GF08', 'user_query': 'Muestralo en grafico', 'intent': 'follow_up_visualization', 'tools': 'render_chart_data', 'requires_clarification': False, 'ideal_response': 'transformar ultima respuesta a chart', 'expected_caveat': 'chart reflects deterministic output'},
    {'flow_id': 'GF09', 'user_query': 'Que problemas tiene Bogota esta semana?', 'intent': 'insight_request', 'tools': 'run_insight_detectors', 'requires_clarification': False, 'ideal_response': 'top insights por final_rank_score', 'expected_caveat': 'thresholds provisionales'},
    {'flow_id': 'GF10', 'user_query': 'Demuestrame que Restaurants SS > ATC CVR causa mas orders', 'intent': 'hypothesis_request', 'tools': 'generate_hypothesis_candidates', 'requires_clarification': True, 'ideal_response': 'rechazar causalidad fuerte, ofrecer evidencia asociativa', 'expected_caveat': 'no causal claims'},
])

display(golden_flows)